In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import glob

# Configuration
# List of files to look for. You can also use glob.glob("results_P*.txt")
files = [
    "results_P1.txt", "results_P2.txt", "results_P4.txt", 
    "results_P8.txt", "results_P16.txt", "results_P32.txt", 
    "results_P64.txt", "results_P128.txt", "results_P256.txt", 
    "results_P512.txt",  "results_P1024.txt"
]

def parse_spmv_log(filename):
    """Parses a single result file for execution metrics."""
    metrics = []
    rank_times = {} # specific for P=64 case
    
    if not os.path.exists(filename):
        print(f"Warning: {filename} not found. Skipping.")
        return None, None

    with open(filename, 'r') as f:
        content = f.read()

    # Extract blocks for each run (Assuming 5 runs)
    # We look for "Max Time:" lines
    max_times = [float(x) for x in re.findall(r"Max Time:\s+([0-9.]+)\s+s", content)]
    min_times = [float(x) for x in re.findall(r"Min Time:\s+([0-9.]+)\s+s", content)]
    avg_times = [float(x) for x in re.findall(r"Avg Time:\s+([0-9.]+)\s+s", content)]
    imbalances = [float(x) for x in re.findall(r"Load Imbalance:\s+([0-9.]+)", content)]
    
    # Extract Processor count
    proc_match = re.search(r"Processors:\s+(\d+)", content)
    if not proc_match:
        return None, None
    p = int(proc_match.group(1))

    # Check if we have valid data
    if not max_times:
        print(f"Warning: No valid run data found in {filename} (Job likely failed).")
        return None, None

    # Create a small dataframe for the runs in this file
    df_runs = pd.DataFrame({
        'Max Time': max_times,
        'Min Time': min_times,
        'Avg Time': avg_times,
        'Load Imbalance': imbalances
    })

    # Average the runs
    avg_data = df_runs.mean().to_dict()
    avg_data['Processors'] = p
    
    # Extract Per-Process Times if available (Specific for P=64 requirement)
    if "Per-Process Times" in content:
        # Find the block
        block = content.split("--- Per-Process Times (for plotting) ---")[-1]
        # Extract Rank X: Time
        matches = re.findall(r"Rank\s+(\d+):\s+([0-9.]+)", block)
        for rank, time in matches:
            rank_times[int(rank)] = float(time)
            
    return avg_data, rank_times

def main():
    results = []
    p64_distribution = {}

    print("Parsing files...")
    for fname in files:
        data, rank_data = parse_spmv_log(fname)
        if data:
            results.append(data)
        if rank_data and data['Processors'] == 64:
            p64_distribution = rank_data

    if not results:
        print("No valid results found.")
        return

    # Create Main DataFrame
    df = pd.DataFrame(results)
    df = df.sort_values('Processors').reset_index(drop=True)

    # Calculate Scaling Metrics
    # T_1 is the time of the first row (Processors = 1)
    try:
        t1 = df[df['Processors'] == 1]['Max Time'].iloc[0]
    except IndexError:
        print("Error: Results for 1 processor not found. Cannot calculate speedup.")
        return

    df['Speedup'] = t1 / df['Max Time']
    df['Efficiency'] = df['Speedup'] / df['Processors']
    df['Ideal Speedup'] = df['Processors']

    # --- DISPLAY TABLE ---
    print("\n=== Performance Summary Table ===")
    print(df[['Processors', 'Max Time', 'Min Time', 'Speedup', 'Efficiency', 'Load Imbalance']].to_string(index=False, float_format="%.4f"))

    # Save Table to CSV
    df.to_csv('spmv_performance_metrics.csv', index=False)
    print("\nTable saved to 'spmv_performance_metrics.csv'")

    # --- PLOTTING ---
    plt.style.use('seaborn-v0_8-whitegrid') # Use a nice style
    
    # 1. Execution Time vs Processors
    plt.figure(figsize=(10, 6))
    plt.plot(df['Processors'], df['Max Time'], 'o-', linewidth=2, label='Total Execution Time (Max)')
    plt.plot(df['Processors'], df['Min Time'], 's--', linewidth=1, alpha=0.7, label='Computation Time (Min)')
    plt.xlabel('Number of MPI Processes')
    plt.ylabel('Time (seconds)')
    plt.title('Execution Time vs Process Count')
    plt.legend()
    plt.xscale('log', base=2)
    plt.yscale('log')
    plt.xticks(df['Processors'], df['Processors'])
    plt.grid(True, which="both", ls="-", alpha=0.3)
    plt.savefig('plot_execution_time.png')
    print("Generated 'plot_execution_time.png'")

    # 2. Speedup
    plt.figure(figsize=(10, 6))
    plt.plot(df['Processors'], df['Speedup'], 'o-', linewidth=2, color='green', label='Measured Speedup')
    plt.plot(df['Processors'], df['Ideal Speedup'], '--', color='gray', label='Ideal Speedup')
    plt.xlabel('Number of MPI Processes')
    plt.ylabel('Speedup (T1 / Tp)')
    plt.title('Strong Scaling Speedup')
    plt.legend()
    plt.xscale('log', base=2)
    plt.yscale('log', base=2)
    plt.xticks(df['Processors'], df['Processors'])
    plt.yticks(df['Processors'], df['Processors']) # Optional: match y-ticks to p-counts
    plt.grid(True, which="both", ls="-", alpha=0.3)
    plt.savefig('plot_speedup.png')
    print("Generated 'plot_speedup.png'")

    # 3. Load Imbalance
    plt.figure(figsize=(10, 6))
    plt.plot(df['Processors'], df['Load Imbalance'], 'D-', color='red', linewidth=2)
    plt.xlabel('Number of MPI Processes')
    plt.ylabel('Load Imbalance Metric')
    plt.title('Load Imbalance vs Process Count')
    plt.xscale('log', base=2)
    plt.xticks(df['Processors'], df['Processors'])
    plt.grid(True)
    plt.savefig('plot_load_imbalance.png')
    print("Generated 'plot_load_imbalance.png'")

    # 4. P=64 Computation Distribution
    if p64_distribution:
        ranks = sorted(p64_distribution.keys())
        times = [p64_distribution[r] for r in ranks]
        
        plt.figure(figsize=(12, 6))
        plt.bar(ranks, times, color='skyblue', edgecolor='black', width=0.8)
        plt.xlabel('MPI Rank ID')
        plt.ylabel('Computation Time (s)')
        plt.title('Computation Time Distribution (P=64)')
        plt.xlim(-1, 64)
        
        # Add average line
        avg_64 = np.mean(times)
        plt.axhline(y=avg_64, color='r', linestyle='--', label=f'Avg: {avg_64:.4f}s')
        plt.legend()
        
        plt.savefig('plot_p64_distribution.png')
        print("Generated 'plot_p64_distribution.png'")
    else:
        print("Warning: P=64 distribution data not found (check results_P64.txt).")

    plt.close('all')

if __name__ == "__main__":
    main()